# Notebook 12: external baselines on the same folds

Every method below uses the identical 5 lesion-level folds, the identical inner validation split, and the same metrics as notebooks 07/08.

| Method | Encoder | Fusion | Trained with missing inputs? |
|---|---|---|---|
| PanDerm-LP (D), (C) | PanDerm ViT-L/16, frozen | none, logistic regression | n/a |
| PanDerm-LP (C+D+M) | PanDerm ViT-L/16, frozen | feature concatenation, logistic regression | no (missing blocks mean-imputed) |
| PanDerm + our fusion | PanDerm ViT-L/16, frozen | our late fusion + modality dropout | yes |
| Concat-MLP | our v2 ConvNeXtV2 | early concatenation MLP | no (mean-imputed) |
| Gated fusion (DVM-SLC style) | our v2 ConvNeXtV2 | learned softmax gate over modality tokens | no (mean-imputed) |
| Ours | our v2 ConvNeXtV2 | late fusion + modality dropout | yes |

The first part (PanDerm features) is the only GPU-heavy step, around 10 to 15 minutes on a T4. Everything after it trains small heads.

**Inputs:** MILK10_DATASET, the v2 seed-42 training notebook (stage2v2_s42), GPU T4, Internet on (for the PanDerm repo and weights). If Google Drive refuses the weight download, upload `panderm_ll_data6_checkpoint-499.pth` as a Kaggle dataset and attach it; the notebook finds it automatically.

In [ ]:
!pip install -q gdown open_clip_torch
!test -d /kaggle/working/PanDerm || git clone -q --depth 1 https://github.com/SiyuanYan1/PanDerm /kaggle/working/PanDerm

In [ ]:
import os, sys, time, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import torch, torch.nn as nn, torch.nn.functional as F

INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input"); WORK = os.environ.get("KAGGLE_WORK", "/kaggle/working")
OUT = f"{WORK}/stage12"; os.makedirs(OUT, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device)
SEED = 42
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
MODS = ["C", "D", "M"]; PATTERNS = ["CDM", "CD", "CM", "DM", "C", "D", "M"]
ACTIONABLE = ["AKIEC", "BCC", "MAL_OTH", "MEL", "SCCKA"]; is_act_cls = np.isin(CLASSES, ACTIONABLE)
REPORT_PATS = ["CDM", "CM", "C", "D"]

STAGE2 = None
for root, dirs, files in os.walk(INPUT, followlinks=True):
    if os.path.basename(root) == "stage2v2_s42" and "emb_D_fold0.npy" in files: STAGE2 = root; break
assert STAGE2, "attach the v2 seed-42 training notebook (folder stage2v2_s42)"
les = pd.read_csv(f"{STAGE2}/milk10k_lesions_folds.csv")
y_all = y = les["y"].values.astype(int); fold = les["fold"].values; n = len(les)
y_act = is_act_cls[y]
tone = les["tone_group"].values
MONET = [c[:-2] for c in les.columns if c.startswith("MONET_") and c.endswith("_C")]
print("v2 run:", STAGE2, "| lesions:", n)

# raw training images (PanDerm uses its own preprocessing, so no Shades of Gray here)
want = set(les["isic_id_C"]) | set(les["isic_id_D"]); IMG = {}
for root, dirs, files in os.walk(INPUT, followlinks=True):
    for q in files:
        s = os.path.splitext(q)[0]
        if s in want and q.lower().endswith(".jpg"): IMG.setdefault(s, os.path.join(root, q))
assert len(IMG) == len(want), f"found {len(IMG)} of {len(want)} training images: attach MILK10_DATASET"

CKPT_NAME = "panderm_ll_data6_checkpoint-499.pth"
CKPT = None
for root, dirs, files in os.walk(INPUT, followlinks=True):
    if CKPT_NAME in files: CKPT = os.path.join(root, CKPT_NAME); break
if CKPT is None:
    CKPT = f"{WORK}/{CKPT_NAME}"
    if not os.path.exists(CKPT):
        import gdown
        gdown.download(id="1SwEzaOlFV_gBKf2UzeowMC8z9UH7AQbE", output=CKPT, quiet=False)
assert os.path.exists(CKPT) and os.path.getsize(CKPT) > 1e8, "PanDerm weights missing: upload the .pth as a dataset"
print("PanDerm checkpoint:", CKPT, f"{os.path.getsize(CKPT) / 1e9:.2f} GB")

In [ ]:
# ---- PanDerm-Large frozen features (CLS token, 1024-d), the repo's own linear-probe setup ----
sys.path.insert(0, f"{WORK}/PanDerm/classification")
from models.modeling_finetune import panderm_large_patch16_224
from models.builder import get_eval_transforms
from torch.utils.data import Dataset, DataLoader

def load_panderm():
    model = panderm_large_patch16_224()
    sd = torch.load(CKPT, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and "model" in sd and not any(k.startswith("encoder.") for k in sd): sd = sd["model"]
    sd = {k.replace("encoder.", ""): v for k, v in sd.items()}
    msg = model.load_state_dict(sd, strict=False)
    own = set(model.state_dict()); loaded = own & set(sd)
    print(f"PanDerm weights: {len(loaded)}/{len(own)} tensors loaded | missing e.g. {sorted(msg.missing_keys)[:5]}")
    assert len(loaded) / len(own) > 0.95, "checkpoint did not load: wrong file?"
    return model.to(device).eval()

class Raw(Dataset):
    def __init__(self, ids, tf): self.ids, self.tf = list(ids), tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.tf(Image.open(IMG[self.ids[i]]).convert("RGB"))

PD = {}
for m in ["C", "D"]:
    dst = f"{OUT}/panderm_{m}.npy"
    if os.path.exists(dst): PD[m] = np.load(dst); continue
    if "pd_model" not in globals(): pd_model = load_panderm(); tf = get_eval_transforms("imagenet", 256, True)
    dl = DataLoader(Raw(les[f"isic_id_{m}"], tf), batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
    feats, t0 = [], time.time()
    with torch.no_grad():
        for x in dl:
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                feats.append(pd_model.forward_features(x.to(device), is_train=False).float().cpu().numpy())
    PD[m] = np.concatenate(feats); np.save(dst, PD[m]); print(m, PD[m].shape, f"{time.time() - t0:.0f}s")
if "pd_model" in globals(): del pd_model; torch.cuda.empty_cache()

In [ ]:
class FusionNet(nn.Module):
    """Late fusion head from notebook 03/08 (kind='late' is the paper model)."""
    def __init__(self, dims, kind="late", d=256, drop=0.3):
        super().__init__()
        self.kind = kind
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Dropout(0.2 if m != "M" else 0.0), nn.Linear(dims[m], d),
                                                    nn.GELU(), nn.LayerNorm(d)) for m in MODS})
        self.heads = nn.ModuleDict({m: nn.Sequential(nn.Dropout(drop), nn.Linear(d, K)) for m in MODS})
    def forward(self, x, avail):                      # avail: (B,3) bool in MODS order
        tok = torch.stack([self.proj[m](x[m]) for m in MODS], 1)
        logits = torch.stack([self.heads[m](tok[:, i]) for i, m in enumerate(MODS)], 1)
        w = avail.float().unsqueeze(-1)
        return (logits * w).sum(1) / w.sum(1).clamp(min=1)

def pattern_mask(p, n):
    return torch.tensor([[m in p for m in MODS]] * n, dtype=torch.bool, device=device)
NONEMPTY = torch.tensor([[m in p for m in MODS] for p in PATTERNS], dtype=torch.bool)
def sample_avail(n, mode, fixed=None):
    if mode == "fixed": return pattern_mask(fixed, n)
    full = torch.ones(n, 3, dtype=torch.bool)
    if mode == "none": return full.to(device)
    sub = NONEMPTY[torch.randint(len(PATTERNS), (n,))]
    keep_full = torch.rand(n) < 0.5
    return torch.where(keep_full.unsqueeze(1), full, sub).to(device)

def evaluate(y, P):
    pred = P.argmax(1)
    out = {"bal_acc": balanced_accuracy_score(y, pred),
           "macro_f1": f1_score(y, pred, average="macro", labels=range(K), zero_division=0),
           "macro_f1_thr05": np.mean([f1_score(y == k, P[:, k] >= 0.5, zero_division=0) for k in range(K)])}
    try: out["macro_auc"] = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=range(K))
    except ValueError: out["macro_auc"] = np.nan
    return out

def to_t(X, idx): return {m: torch.tensor(X[m][idx], device=device) for m in MODS}

@torch.no_grad()
def predict(model, X, idx, pattern):
    model.eval(); x = to_t(X, idx)
    return model(x, pattern_mask(pattern, len(idx))).float().softmax(1).cpu().numpy()

def backbone_val_ids(f):
    """Inner 10% split used for checkpoint selection in notebooks 02/07 (out-of-sample embeddings)."""
    trval = les[les["fold"] != f]
    _, va = train_test_split(trval, test_size=0.10, stratify=trval["y"], random_state=SEED)
    return les.index.isin(va.index)

def train_fusion(X, f, mode="dropout", fixed=None, build=None, epochs=80, patience=15, bs=256, lr=3e-4):
    """Same recipe as notebook 08. Rows 0..len(les)-1 of X are the training lesions; extra rows (benchmark) are never touched."""
    torch.manual_seed(SEED + f); np.random.seed(SEED + f)
    is_val = backbone_val_ids(f)
    tr_idx = np.where((les["fold"].values != f) & ~is_val)[0]
    va_idx = np.where(is_val)[0]
    n_ = np.bincount(y_all[tr_idx], minlength=K).astype(float)
    w = np.sqrt(n_.max() / np.maximum(n_, 1)); w = torch.tensor(w / w.mean(), dtype=torch.float, device=device)
    dims = {m: X[m].shape[1] for m in MODS}
    model = (build(dims) if build else FusionNet(dims, "late")).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    Xtr, ytr = to_t(X, tr_idx), torch.tensor(y_all[tr_idx], device=device)
    best, bad, state = -1, 0, None
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(tr_idx), device=device)
        for s in range(0, len(perm), bs):
            b = perm[s:s + bs]
            xb = {m: Xtr[m][b] for m in MODS}
            loss = F.cross_entropy(model(xb, sample_avail(len(b), mode, fixed)), ytr[b], weight=w, label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step()
        pats = PATTERNS if mode == "dropout" else [fixed if mode == "fixed" else "CDM"]
        score = np.mean([evaluate(y_all[va_idx], predict(model, X, va_idx, p))["macro_f1"] for p in pats])
        if score > best: best, bad, state = score, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state)
    return model, ep + 1

SITE_CATS = sorted(les["site"].fillna("unknown").unique())
def meta_matrix(df):
    site = pd.Categorical(df["site"].fillna("unknown"), categories=SITE_CATS)
    return np.column_stack([df["age_approx"].fillna(-1).values, df["age_approx"].isna().values,
                            df["skin_tone_class"].values, (df["sex"] == "male").values,
                            pd.get_dummies(site).values]).astype(np.float32)

def standardise(X, tr_mask):
    """Fill missing age with the training median, then z-score every block with TRAIN lesions of this fold only."""
    X = {m: v.copy() for m, v in X.items()}
    age = X["M"][:, 0]; age[age < 0] = np.median(age[tr_mask & (age >= 0)])
    for m in MODS:
        mu, sd = X[m][tr_mask].mean(0), X[m][tr_mask].std(0) + 1e-6
        X[m] = (X[m] - mu) / sd
    return X

In [ ]:
META_ALL = meta_matrix(les)
def inputs(f, enc):
    """enc='v2': our ConvNeXtV2 embeddings (+MONET), enc='panderm': PanDerm features (+MONET). Same metadata block."""
    X = {}
    for m in ["C", "D"]:
        E = np.load(f"{STAGE2}/emb_{m}_fold{f}.npy").astype(np.float32) if enc == "v2" else PD[m].astype(np.float32)
        X[m] = np.concatenate([E, les[[f"{c}_{m}" for c in MONET]].values.astype(np.float32)], 1)
    X["M"] = META_ALL.copy()
    return standardise(X, fold != f)

# ---- baseline heads (both trained with all inputs present, as published; missing inputs are mean-imputed at test time) ----
class ConcatMLP(nn.Module):
    def __init__(self, dims, d=256, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(0.2), nn.Linear(sum(dims.values()), 2 * d), nn.GELU(), nn.LayerNorm(2 * d),
                                 nn.Dropout(drop), nn.Linear(2 * d, K))
    def forward(self, x, avail):
        return self.net(torch.cat([x[m] * avail[:, i:i + 1] for i, m in enumerate(MODS)], 1))

class GatedFusion(nn.Module):
    """Modality tokens weighted by a softmax gate computed from all tokens (DVM-SLC-style decision gate)."""
    def __init__(self, dims, d=256, drop=0.3):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Dropout(0.2 if m != "M" else 0.0), nn.Linear(dims[m], d),
                                                    nn.GELU(), nn.LayerNorm(d)) for m in MODS})
        self.gate = nn.Sequential(nn.Linear(3 * d, d), nn.GELU(), nn.Linear(d, 3))
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(d, K))
        self.last_gate = None
    def forward(self, x, avail):
        tok = torch.stack([self.proj[m](x[m] * avail[:, i:i + 1]) for i, m in enumerate(MODS)], 1)   # mean-imputed input
        g = self.gate(tok.flatten(1)).softmax(1); self.last_gate = g.detach()
        return self.head((g.unsqueeze(-1) * tok).sum(1))

In [ ]:
# ---- PanDerm linear probe: logistic regression, C picked on the inner validation split, refit on all training lesions ----
C_GRID = [0.001, 0.01, 0.1, 1.0]
def lp_fit_predict(X, f, blocks):
    """blocks: modalities concatenated into the probe. Missing blocks at test time are set to 0 (= training mean)."""
    is_val = backbone_val_ids(f); trv = fold != f; tr = trv & ~is_val; te = np.where(fold == f)[0]
    Z = lambda rows, avail=blocks: np.concatenate([X[m][rows] if m in avail else np.zeros_like(X[m][rows]) for m in blocks], 1)
    best = (-1, None)
    for c in C_GRID:
        lr = LogisticRegression(C=c, max_iter=3000, class_weight="balanced").fit(Z(tr), y[tr])
        s = f1_score(y[is_val], lr.predict(Z(is_val)), average="macro", labels=range(K), zero_division=0)
        if s > best[0]: best = (s, c)
    lr = LogisticRegression(C=best[1], max_iter=3000, class_weight="balanced").fit(Z(trv), y[trv])
    out = {}
    for p in (REPORT_PATS if len(blocks) > 1 else blocks):   # a single-image probe is only reported on its own input
        av = [m for m in blocks if m in p]
        if av: out[p] = (te, lr.predict_proba(Z(te, av)))
    return out, best[1]

PROBS = {}            # (method, pattern) -> OOF probabilities
def put(method, p, te, P):
    PROBS.setdefault((method, p), np.full((n, K), np.nan, np.float32))[te] = P

t0 = time.time(); chosenC = []
for f in range(5):
    Xp = inputs(f, "panderm")
    for name, blocks in [("PanDerm-LP (D)", ["D"]), ("PanDerm-LP (C)", ["C"]), ("PanDerm-LP (C+D+M)", ["C", "D", "M"])]:
        res, c = lp_fit_predict(Xp, f, blocks); chosenC.append(dict(method=name, fold=f, C=c))
        for p, (te, P) in res.items(): put(name, p, te, P)
    print(f"fold {f} probes done | {(time.time() - t0) / 60:.1f} min")
pd.DataFrame(chosenC).to_csv(f"{OUT}/panderm_lp_chosen_C.csv", index=False)

In [ ]:
# ---- neural heads: PanDerm + our fusion, concat-MLP, gated fusion, ours (all on seed-42 folds) ----
HEADS = {
    "PanDerm + our fusion": dict(enc="panderm", mode="dropout", build=None),
    "Concat-MLP":           dict(enc="v2", mode="none", build=lambda d: ConcatMLP(d)),
    "Gated fusion":         dict(enc="v2", mode="none", build=lambda d: GatedFusion(d)),
    "Ours (late fusion + modality dropout)": dict(enc="v2", mode="dropout", build=None),
}
GATES = []
t0 = time.time()
for f in range(5):
    te = np.where(fold == f)[0]
    X = {"v2": inputs(f, "v2"), "panderm": inputs(f, "panderm")}
    for name, cfg in HEADS.items():
        model, _ = train_fusion(X[cfg["enc"]], f, mode=cfg["mode"], build=cfg["build"])
        for p in REPORT_PATS:
            put(name, p, te, predict(model, X[cfg["enc"]], te, p))
            if name == "Gated fusion": GATES.append(dict(fold=f, pattern=p, **dict(zip(["w_C", "w_D", "w_M"], model.last_gate.mean(0).cpu().numpy()))))
        del model
    print(f"fold {f} heads done | {(time.time() - t0) / 60:.1f} min")
pd.DataFrame(GATES).groupby("pattern")[["w_C", "w_D", "w_M"]].mean().round(3)

In [ ]:
# ---- metrics: per-fold mean +- SD and pooled OOF ----
TARGET_SPEC = 0.70
def act_sens_at_spec(P):
    s = P[:, is_act_cls].sum(1); flag = np.zeros(n, bool)
    for f in range(5):
        te = fold == f; flag[te] = s[te] >= np.quantile(s[(fold != f) & ~y_act], TARGET_SPEC)
    return (flag & y_act).sum() / y_act.sum()

def all_metrics(P, idx=None):
    idx = np.arange(n) if idx is None else idx
    e = evaluate(y[idx], P[idx])
    e["MEL_recall"] = recall_score(y[idx] == 7, P[idx].argmax(1) == 7, zero_division=0)
    e["act_AUROC"] = roc_auc_score(y_act[idx], P[idx][:, is_act_cls].sum(1))
    return e

rows = []
for (method, p), P in PROBS.items():
    if np.isnan(P).any(): continue
    for f in range(5):
        rows.append(dict(method=method, pattern=p, fold=f, **all_metrics(P, np.where(fold == f)[0])))
per_fold = pd.DataFrame(rows); per_fold.to_csv(f"{OUT}/baselines_per_fold.csv", index=False)

pooled = pd.DataFrame([dict(method=m, pattern=p, **all_metrics(P), act_sens_at_spec70=act_sens_at_spec(P))
                       for (m, p), P in PROBS.items() if not np.isnan(P).any()])
pooled.to_csv(f"{OUT}/baselines_pooled.csv", index=False)

ORDER = ["PanDerm-LP (D)", "PanDerm-LP (C)", "PanDerm-LP (C+D+M)", "PanDerm + our fusion", "Concat-MLP", "Gated fusion",
         "Ours (late fusion + modality dropout)"]
agg = per_fold.groupby(["method", "pattern"])[["macro_f1", "bal_acc", "macro_auc", "MEL_recall", "act_AUROC"]].agg(["mean", "std"])
fmt = pd.DataFrame({c: agg[(c, "mean")].round(3).astype(str) + " ± " + agg[(c, "std")].round(3).astype(str)
                    for c in ["macro_f1", "bal_acc", "macro_auc", "MEL_recall", "act_AUROC"]})
fmt = fmt.join(pooled.set_index(["method", "pattern"])["act_sens_at_spec70"].round(3))
fmt = fmt.reindex([(m, p) for m in ORDER for p in REPORT_PATS if (m, p) in fmt.index])
fmt.to_csv(f"{OUT}/TABLE_external_baselines.csv"); fmt

In [ ]:
# ---- missing-modality view: macro F1 by input pattern (rows = method) ----
wide = pooled.pivot(index="method", columns="pattern", values="macro_f1").reindex(ORDER)[REPORT_PATS]
wide["drop_CDM_to_CM"] = wide["CDM"] - wide["CM"]
wide.to_csv(f"{OUT}/TABLE_baselines_by_pattern_macroF1.csv"); wide.round(3)

In [ ]:
# ---- paired lesion-level bootstrap: ours minus each baseline (same lesions, same folds) ----
OURS = "Ours (late fusion + modality dropout)"; B = 2000; rng = np.random.default_rng(0)
def mf1(yy, P): return f1_score(yy, P.argmax(1), average="macro", labels=range(K), zero_division=0)
boot = []
for p in ["CDM", "CM"]:
    Po = PROBS[(OURS, p)]
    for m in ORDER[:-1]:
        if (m, p) not in PROBS: continue
        Pb = PROBS[(m, p)]; d0 = mf1(y, Po) - mf1(y, Pb); ds = []
        for _ in range(B):
            i = rng.integers(0, n, n); ds.append(mf1(y[i], Po[i]) - mf1(y[i], Pb[i]))
        lo, hi = np.percentile(ds, [2.5, 97.5])
        wins = int((per_fold[(per_fold.method == OURS) & (per_fold.pattern == p)]["macro_f1"].values >
                    per_fold[(per_fold.method == m) & (per_fold.pattern == p)]["macro_f1"].values).sum())
        boot.append(dict(pattern=p, baseline=m, delta_macroF1=d0, ci_low=lo, ci_high=hi, p_le_0=np.mean(np.array(ds) <= 0),
                         folds_won=f"{wins}/5"))
boot = pd.DataFrame(boot); boot.to_csv(f"{OUT}/TABLE_paired_bootstrap.csv", index=False); boot.round(4)

In [ ]:
# ---- figure: macro F1 by input pattern (greyscale, print-safe) ----
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3.6))
greys = ["0.9", "0.75", "0.6", "0.45", "0.3", "0.15", "0.0"]; hatches = ["", "//", "..", "xx", "\\\\", "--", ""]
w_ = 0.12; x0 = np.arange(len(REPORT_PATS))
for i, m in enumerate(ORDER):
    vals = [wide.loc[m, p] if p in wide.columns and not pd.isna(wide.loc[m, p]) else 0 for p in REPORT_PATS]
    ax.bar(x0 + (i - 3) * w_, vals, w_, color=greys[i], edgecolor="black", hatch=hatches[i], linewidth=0.6, label=m)
ax.set_xticks(x0); ax.set_xticklabels(["full (C+D+M)", "no dermoscopy (C+M)", "clinical only", "dermoscopy only"])
ax.set_ylabel("macro F1 (OOF)"); ax.legend(fontsize=7, ncol=2, frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT}/F_external_baselines.png", dpi=300); plt.show()

## Notes for the paper

* PanDerm is used exactly as its authors' linear-probe code sets it up (frozen ViT-L/16, CLS token, Resize 256 + CenterCrop 224, ImageNet normalisation). Its pretraining corpus includes public ISIC Archive images, so a small overlap with MILK10k cannot be ruled out; state this as a limitation. If anything it favours the baseline.
* "Gated fusion" is a re-implementation of a learned decision gate in the spirit of DVM-SLC, not the authors' code. Call it "DVM-SLC-style gated fusion" in the text.
* "PanDerm + our fusion" separates the encoder from the fusion recipe: if it beats PanDerm-LP (C+D+M) under missing inputs, the robustness comes from modality dropout and not from our encoder.